# Getting Started
Antes de mais nada precisamos definir as configuracoes de conexao com o git(necessario Token). <br>
Para isso, existe o gitConnection para gerenciar autenticação, leitura e escrita no repositório.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from gitlake import GitConnection

REPO_URL = os.getenv("REPO_URL")
GIT_TOKEN = os.getenv("GIT_TOKEN")

# Instancia a conexão com o GitHub
git = GitConnection(
    repo_url=REPO_URL,
    username="gitlake",
    token=GIT_TOKEN,  # <- Agora usa o valor real da variável
    branch="main",
)

# Gerenciando coleções
Para gerenciar nossas coleções de dados, existe o CollectionManager. <br>
Ele permite criar, listar, salvar, ler e excluir coleções de DataFrames armazenadas no GitHub, mantendo um registro central (collections_registry.json).

In [ ]:
from gitlake import Collection, CollectionManager

cm = CollectionManager(git) #Falamos qual conexao o CollectionManager vai conectar para gerenciar as colecoes.

# Cria um novo Collection local
concessionaria_veiculos_seminovos = Collection(
    name="veiculos_seminovos",
    base_path="concessionaria",
    path="veiculos/seminovos",
    format="parquet"
)

# Registra a collection no repositorio
if cm.create_collection(concessionaria_veiculos_seminovos):
    print("Coleção criada")
else:
    print("A coleção já existe")

# Persistindo e lendo coleções

In [ ]:
import pandas as pd

# Criando o DataFrame de vendas de veículos
df = pd.DataFrame({
    "Modelo": ["Gol", "Civic", "Corolla", "Onix", "Hilux"],
    "Ano": [2018, 2020, 2019, 2021, 2017],
    "KM": [45000, 30000, 40000, 15000, 60000],
    "Preco": [45000, 95000, 88000, 70000, 155000]
})

#Salva o dataframe usando save_dataframe do CollectionManager
cm.save_dataframe(df=df, collection_name="veiculos_seminovos", mode="overwrite")

In [ ]:
#Lendo a colecao:
df = cm.read_dataframe("veiculos_seminovos")
print(df)

# Removendo dados e coleções
Existem dois níveis de remoção:
- `delete_dataframe`: sobrescreve os dados da coleção com um DataFrame vazio, mas mantém a coleção registrada.
- `delete_collection`: remove o arquivo de dados do GitHub **e** o registro da coleção em `collections_registry.json`.

In [ ]:
# Limpa os dados, mas mantém a coleção registrada
cm.delete_dataframe("veiculos_seminovos")

df = cm.read_dataframe("veiculos_seminovos")
print(df)  # DataFrame vazio
print("Coleção ainda registrada:", cm.collection_exists("veiculos_seminovos"))

In [ ]:
# Remove a coleção por completo (dados + registro)
cm.delete_collection("veiculos_seminovos")

print("Coleção ainda registrada:", cm.collection_exists("veiculos_seminovos"))